In [ ]:
from neo4j import GraphDatabase
from tqdm import tqdm

# --- Config ---
URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "Prabhu13")  # change this

In [ ]:
import os
import pandas as pd
from neo4j import GraphDatabase
from datetime import datetime
 
# ─────────────────────────────────────────────
# CONFIG — update these
# ─────────────────────────────────────────────
NEO4J_URI      = "bolt://127.0.0.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "xxxx"
 
DATA_DIR       = "input"          # folder with all Olist CSVs
SAMPLE_SIZE    = 1000              # number of customers to sample (None = load all)


 

In [35]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
 

In [36]:
def load_csvs():
    print("📂 Loading CSVs...")
    customers    = pd.read_csv(f"{DATA_DIR}/olist_customers_dataset.csv")
    orders       = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv")
    order_items  = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
    payments     = pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset.csv")
    reviews      = pd.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv")
    products     = pd.read_csv(f"{DATA_DIR}/olist_products_dataset.csv")
    sellers      = pd.read_csv(f"{DATA_DIR}/olist_sellers_dataset.csv")
    translations = pd.read_csv(f"{DATA_DIR}/product_category_name_translation.csv")
 
    # Translate category names to English
    products = products.merge(translations, on="product_category_name", how="left")
    products["category"] = products["product_category_name_english"].fillna(
        products["product_category_name"]
    ).fillna("unknown")
 
    return customers, orders, order_items, payments, reviews, products, sellers

In [37]:
def sample_data(customers, orders, order_items, payments, reviews, products, sellers):
    if SAMPLE_SIZE is None:
        return customers, orders, order_items, payments, reviews, products, sellers
 
    print(f"✂️  Sampling {SAMPLE_SIZE} unique customers...")
    sampled_customers = customers.drop_duplicates("customer_unique_id").sample(
        n=SAMPLE_SIZE, random_state=42
    )
    sampled_customer_ids = set(sampled_customers["customer_id"])
 
    sampled_orders      = orders[orders["customer_id"].isin(sampled_customer_ids)]
    sampled_order_ids   = set(sampled_orders["order_id"])
 
    sampled_items    = order_items[order_items["order_id"].isin(sampled_order_ids)]
    sampled_payments = payments[payments["order_id"].isin(sampled_order_ids)]
    sampled_reviews  = reviews[reviews["order_id"].isin(sampled_order_ids)]
 
    sampled_product_ids = set(sampled_items["product_id"])
    sampled_seller_ids  = set(sampled_items["seller_id"])
 
    sampled_products = products[products["product_id"].isin(sampled_product_ids)]
    sampled_sellers  = sellers[sellers["seller_id"].isin(sampled_seller_ids)]
 
    print(f"   customers={len(sampled_customers)}, orders={len(sampled_orders)}, "
          f"items={len(sampled_items)}, products={len(sampled_products)}, "
          f"sellers={len(sampled_sellers)}")
 
    return (sampled_customers, sampled_orders, sampled_items,
            sampled_payments, sampled_reviews, sampled_products, sampled_sellers)

In [39]:
def derive_preferences(customers, orders, payments):
    """
    Preferred payment = payment type used most by each customer.
    Stored as a property on Customer node — Preference Memory.
    """
    print("🧠 Deriving preference memory...")
    order_customer = orders[["order_id", "customer_id"]]
    pay_merged = payments.merge(order_customer, on="order_id")
    pay_merged = pay_merged.merge(
        customers[["customer_id", "customer_unique_id"]], on="customer_id"
    )
    preferred = (
        pay_merged.groupby(["customer_unique_id", "payment_type"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .drop_duplicates("customer_unique_id")
        .rename(columns={"payment_type": "preferred_payment"})
    )
    return preferred[["customer_unique_id", "preferred_payment"]]
 
 
def derive_emotional_label(score):
    """Map review score to emotional label — Emotional Memory."""
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative"
 
 

In [40]:
# ─────────────────────────────────────────────
# STEP 4 — CREATE CONSTRAINTS & INDEXES
# ─────────────────────────────────────────────
def create_constraints(tx):
    constraints = [
        "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Customer)      REQUIRE c.unique_id  IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (o:Order)         REQUIRE o.order_id   IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Product)       REQUIRE p.product_id IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Seller)        REQUIRE s.seller_id  IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (r:Review)        REQUIRE r.review_id  IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (cat:Category)    REQUIRE cat.name     IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (pm:PaymentMethod) REQUIRE pm.type     IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (city:City)       REQUIRE city.name    IS UNIQUE",
    ]
    for c in constraints:
        tx.run(c)
 

In [41]:
# ─────────────────────────────────────────────
# STEP 5 — NODE CREATION FUNCTIONS
# ─────────────────────────────────────────────
def create_customers(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MERGE (c:Customer {unique_id: row.customer_unique_id})
        SET   c.zip_code          = row.customer_zip_code_prefix,
              c.city              = row.customer_city,
              c.state             = row.customer_state,
              c.preferred_payment = row.preferred_payment
    """, rows=rows)
 
 
def create_orders(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MERGE (o:Order {order_id: row.order_id})
        SET   o.status                        = row.order_status,
              o.purchase_timestamp            = row.order_purchase_timestamp,
              o.approved_at                   = row.order_approved_at,
              o.delivered_carrier_date        = row.order_delivered_carrier_date,
              o.delivered_customer_date       = row.order_delivered_customer_date,
              o.estimated_delivery_date       = row.order_estimated_delivery_date,
              o.year                          = row.year,
              o.month                         = row.month
    """, rows=rows)
 
 
def create_products(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MERGE (p:Product {product_id: row.product_id})
        SET   p.category          = row.category,
              p.weight_g          = row.product_weight_g,
              p.photos_qty        = row.product_photos_qty
    """, rows=rows)
 
 
def create_sellers(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MERGE (s:Seller {seller_id: row.seller_id})
        SET   s.zip_code = row.seller_zip_code_prefix,
              s.city     = row.seller_city,
              s.state    = row.seller_state
    """, rows=rows)
 
 
def create_categories(tx, names):
    tx.run("""
        UNWIND $names AS name
        MERGE (:Category {name: name})
    """, names=names)
 
 
def create_payment_methods(tx, types):
    tx.run("""
        UNWIND $types AS t
        MERGE (:PaymentMethod {type: t})
    """, types=types)
 
 
def create_cities(tx, cities):
    tx.run("""
        UNWIND $cities AS c
        MERGE (:City {name: c})
    """, cities=cities)
 
 
# ─────────────────────────────────────────────
# STEP 6 — RELATIONSHIP CREATION FUNCTIONS
# ─────────────────────────────────────────────
def create_placed_relationships(tx, rows):
    """(Customer)-[:PLACED]->(Order)"""
    tx.run("""
        UNWIND $rows AS row
        MATCH (c:Customer {unique_id: row.customer_unique_id})
        MATCH (o:Order    {order_id:  row.order_id})
        MERGE (c)-[:PLACED]->(o)
    """, rows=rows)
 
 
def create_order_item_relationships(tx, rows):
    """
    (Order)-[:HAS_ITEM]->(OrderItem)-[:IS_PRODUCT]->(Product)
    (OrderItem)-[:FULFILLED_BY]->(Seller)
    """
    tx.run("""
        UNWIND $rows AS row
        MATCH (o:Order   {order_id:   row.order_id})
        MATCH (p:Product {product_id: row.product_id})
        MATCH (s:Seller  {seller_id:  row.seller_id})
        MERGE (oi:OrderItem {item_id: row.item_id})
        SET   oi.price         = row.price,
              oi.freight_value = row.freight_value,
              oi.shipping_limit_date = row.shipping_limit_date
        MERGE (o)-[:HAS_ITEM]->(oi)
        MERGE (oi)-[:IS_PRODUCT]->(p)
        MERGE (oi)-[:FULFILLED_BY]->(s)
    """, rows=rows)
 
 
def create_payment_relationships(tx, rows):
    """(Order)-[:PAID_WITH]->(PaymentMethod)"""
    tx.run("""
        UNWIND $rows AS row
        MATCH (o:Order        {order_id: row.order_id})
        MATCH (pm:PaymentMethod {type:   row.payment_type})
        MERGE (o)-[r:PAID_WITH]->(pm)
        SET   r.installments = row.payment_installments,
              r.value        = row.payment_value,
              r.sequential   = row.payment_sequential
    """, rows=rows)
 
 
def create_review_relationships(tx, rows):
    """
    (Customer)-[:WROTE]->(Review)-[:ABOUT]->(Order)
    Emotional memory: sentiment label on Review node
    """
    tx.run("""
        UNWIND $rows AS row
        MATCH (o:Order {order_id: row.order_id})
        MERGE (r:Review {review_id: row.review_id})
        SET   r.score     = row.review_score,
              r.title     = row.review_comment_title,
              r.sentiment = row.sentiment
        MERGE (r)-[:ABOUT]->(o)
        WITH r, o
        MATCH (c:Customer)-[:PLACED]->(o)
        MERGE (c)-[:WROTE]->(r)
    """, rows=rows)
 
 
def create_category_relationships(tx, rows):
    """(Product)-[:BELONGS_TO]->(Category)"""
    tx.run("""
        UNWIND $rows AS row
        MATCH (p:Product  {product_id: row.product_id})
        MATCH (cat:Category {name:     row.category})
        MERGE (p)-[:BELONGS_TO]->(cat)
    """, rows=rows)
 
 
def create_location_relationships(tx, rows, node_label, id_field):
    """(Customer|Seller)-[:LOCATED_IN]->(City)"""
    tx.run(f"""
        UNWIND $rows AS row
        MATCH (n:{node_label} {{{id_field}: row.id}})
        MATCH (city:City {{name: row.city}})
        MERGE (n)-[:LOCATED_IN]->(city)
    """, rows=rows)
 
 
# ─────────────────────────────────────────────
# STEP 7 — SEMANTIC MEMORY SEEDS
# ─────────────────────────────────────────────
def seed_semantic_memory(tx):
    """
    Canonical world facts — Semantic Memory.
    These are not derived from data; they are known truths.
    """
    facts = [
        # Payment method types
        ("boleto",       "IS_A",        "PaymentInstrument"),
        ("credit_card",  "IS_A",        "PaymentInstrument"),
        ("voucher",      "IS_A",        "PaymentInstrument"),
        ("debit_card",   "IS_A",        "PaymentInstrument"),
        # Brazil geography
        ("SP",           "IS_STATE_OF", "Brazil"),
        ("RJ",           "IS_STATE_OF", "Brazil"),
        ("MG",           "IS_STATE_OF", "Brazil"),
        # Order status semantics
        ("delivered",    "MEANS",       "OrderCompleted"),
        ("canceled",     "MEANS",       "OrderFailed"),
        ("shipped",      "MEANS",       "OrderInTransit"),
    ]
    tx.run("""
        UNWIND $facts AS f
        MERGE (s:SemanticNode {name: f[0]})
        MERGE (o:SemanticNode {name: f[2]})
        MERGE (s)-[:SEMANTIC_RELATION {type: f[1]}]->(o)
    """, facts=facts)
 

In [42]:
# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────
def batch(iterable, size=500):
    lst = list(iterable)
    for i in range(0, len(lst), size):
        yield lst[i:i + size]
 
 
def run_batched(session, fn, rows, size=500):
    total = len(rows)
    for i, chunk in enumerate(batch(rows, size)):
        session.execute_write(fn, chunk)
        print(f"   {min((i+1)*size, total)}/{total}", end="\r")
    print()
 
 

In [43]:
def main():
    # 1. Load
    customers, orders, order_items, payments, reviews, products, sellers = load_csvs()
 
    # 2. Sample
    customers, orders, order_items, payments, reviews, products, sellers = sample_data(
        customers, orders, order_items, payments, reviews, products, sellers
    )
 
    # 3. Derive preference + emotional memory
    preferences = derive_preferences(customers, orders, payments)
    customers = customers.merge(preferences, on="customer_unique_id", how="left")
    customers["preferred_payment"] = customers["preferred_payment"].fillna("unknown")
    reviews["sentiment"] = reviews["review_score"].apply(derive_emotional_label)
 
    # 4. Prepare order temporal fields
    orders["order_purchase_timestamp"] = pd.to_datetime(
        orders["order_purchase_timestamp"], errors="coerce"
    ).astype(str)
    orders["year"]  = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce").dt.year
    orders["month"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce").dt.month
    for col in ["order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"]:
        orders[col] = pd.to_datetime(orders[col], errors="coerce").astype(str)
 
    # 5. Prepare order item IDs
    order_items["item_id"] = order_items["order_id"] + "_" + order_items["order_item_id"].astype(str)
 
    with driver.session() as session:
        # Constraints
        print("🔧 Creating constraints...")
        session.execute_write(create_constraints)
 
        # Canonical nodes — Semantic Memory
        print("🌐 Seeding semantic memory...")
        categories = products["category"].dropna().unique().tolist()
        pay_types  = payments["payment_type"].dropna().unique().tolist()
        all_cities = (
            list(customers["customer_city"].dropna().unique()) +
            list(sellers["seller_city"].dropna().unique())
        )
        session.execute_write(create_categories,      categories)
        session.execute_write(create_payment_methods, pay_types)
        session.execute_write(create_cities,          list(set(all_cities)))
        session.execute_write(seed_semantic_memory)
 
        # Nodes
        print("👤 Creating Customer nodes...")
        run_batched(session, create_customers, customers.to_dict("records"))
 
        print("📦 Creating Order nodes...")
        run_batched(session, create_orders, orders.to_dict("records"))
 
        print("🛍️  Creating Product nodes...")
        run_batched(session, create_products, products.to_dict("records"))
 
        print("🏪 Creating Seller nodes...")
        run_batched(session, create_sellers, sellers.to_dict("records"))
 
        # Relationships
        print("🔗 (Customer)-[:PLACED]->(Order)...")
        order_customer = orders.merge(
            customers[["customer_id", "customer_unique_id"]], on="customer_id"
        )[["customer_unique_id", "order_id"]]
        run_batched(session, create_placed_relationships, order_customer.to_dict("records"))
 
        print("🔗 (Order)-[:HAS_ITEM]->(OrderItem)->(Product/Seller)...")
        run_batched(session, create_order_item_relationships, order_items.to_dict("records"))
 
        print("🔗 (Order)-[:PAID_WITH]->(PaymentMethod)...")
        run_batched(session, create_payment_relationships, payments.to_dict("records"))
 
        print("🔗 (Customer)-[:WROTE]->(Review)-[:ABOUT]->(Order)...")
        run_batched(session, create_review_relationships, reviews.to_dict("records"))
 
        print("🔗 (Product)-[:BELONGS_TO]->(Category)...")
        prod_cat = products[["product_id", "category"]].dropna()
        run_batched(session, create_category_relationships, prod_cat.to_dict("records"))
 
        print("🔗 Location relationships...")
        cust_cities = customers[["customer_unique_id", "customer_city"]].rename(
            columns={"customer_unique_id": "id", "customer_city": "city"}
        )
        sell_cities = sellers[["seller_id", "seller_city"]].rename(
            columns={"seller_id": "id", "seller_city": "city"}
        )
        run_batched(session, lambda tx, rows: create_location_relationships(
            tx, rows, "Customer", "unique_id"), cust_cities.to_dict("records"))
        run_batched(session, lambda tx, rows: create_location_relationships(
            tx, rows, "Seller", "seller_id"), sell_cities.to_dict("records"))
 
    print("\n✅ Done! Knowledge graph loaded into Neo4j.")
    print("\nSample verification queries:")
    print("  MATCH (c:Customer)-[:PLACED]->(o:Order) RETURN count(o) AS total_orders")
    print("  MATCH (c:Customer {preferred_payment:'boleto'}) RETURN count(c)")
    print("  MATCH (r:Review {sentiment:'negative'})-[:ABOUT]->(o:Order) RETURN count(r)")
 
    driver.close()

In [44]:
main()

📂 Loading CSVs...
✂️  Sampling 1000 unique customers...
   customers=1000, orders=1000, items=1121, products=928, sellers=509
🧠 Deriving preference memory...
🔧 Creating constraints...
🌐 Seeding semantic memory...
👤 Creating Customer nodes...
   1000/1000
📦 Creating Order nodes...
   1000/1000
🛍️  Creating Product nodes...
   928/928
🏪 Creating Seller nodes...
   509/509
🔗 (Customer)-[:PLACED]->(Order)...
   1000/1000
🔗 (Order)-[:HAS_ITEM]->(OrderItem)->(Product/Seller)...
   1121/1121
🔗 (Order)-[:PAID_WITH]->(PaymentMethod)...
   1049/1049
🔗 (Customer)-[:WROTE]->(Review)-[:ABOUT]->(Order)...
   993/993
🔗 (Product)-[:BELONGS_TO]->(Category)...
   928/928
🔗 Location relationships...
   1000/1000
   509/509

✅ Done! Knowledge graph loaded into Neo4j.

Sample verification queries:
  MATCH (c:Customer)-[:PLACED]->(o:Order) RETURN count(o) AS total_orders
  MATCH (c:Customer {preferred_payment:'boleto'}) RETURN count(c)
  MATCH (r:Review {sentiment:'negative'})-[:ABOUT]->(o:Order) RETURN cou

In [33]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://127.0.0.1:7687", auth=("neo4j", "Prabhu13"))
driver.verify_connectivity()
print("✅ Connected")
driver.close()

✅ Connected


In [45]:
customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,preferred_payment
0,b48ad91d707041578581aefae1f7c385,b5d6fa3d2213927296ac893f14f4461c,88816,criciuma,SC,credit_card
1,82c5e6b98e498b4150818c913f3e9bc7,0520a11c7af8a73b703f1d2e722c7c8a,19060,presidente prudente,SP,credit_card
2,330a6da5dc55174ac54298ac57c07be1,7a19f3fff09616cbb1cf8fdaa05ae032,12020,taubate,SP,credit_card
3,90caf8b673ecd91ba4242435c5ff5506,dcb605d0a35c2e900646a9014393628d,5414,sao paulo,SP,credit_card
4,6f4fa91813c10c8199496ff532e05ade,372f4d64dacadbf63821f503090ff4de,5641,sao paulo,SP,credit_card
...,...,...,...,...,...,...
995,9a1f57c8665b8cfe232da92bfc983b82,682d203efdcf95d2036a7430a8f32c96,2730,sao paulo,SP,credit_card
996,4d0090e1484f0896bdc470d0b79b5682,48bf4e8301f352f9a5e832e950d53b77,12062,taubate,SP,boleto
997,1a3677b7315ceb6f19ea02af64389b53,a1bb7c7b76a140dfbb6de2b090bf1849,88135,palhoca,SC,credit_card
998,2f7df887ddcb022dae90b092de0e4ef5,b75240cef5e5899733713fc4c9f4206c,35300,caratinga,MG,credit_card
